# Fine-tune FinBERT on Financial PhraseBank

Runs Level 1 fine-tuning: FinBERT (`ProsusAI/finbert`) on the
[Financial PhraseBank](https://huggingface.co/datasets/takala/financial_phrasebank)
dataset (`sentences_allagree` — ~2 264 sentences, full annotator agreement).

The saved model is a drop-in replacement in `SentimentEncoder` and
`SentimentPipeline` via the `model_name_or_path` / `encoder_model` params.

In [ ]:
from sentiment.log import setup_logging
setup_logging()

## Run fine-tuning

In [ ]:
from pathlib import Path
from sentiment.embeddings import fine_tune_finbert

output_dir = fine_tune_finbert(
    Path("../models/finbert-fpb"),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    weight_decay=0.01,
)
print(f"Model saved to: {output_dir}")

## Smoke test

In [ ]:
import numpy as np
from sentiment.embeddings import SentimentEncoder

enc = SentimentEncoder(model_name_or_path=str(output_dir))
label, emb = enc.encode("Apple reports record quarterly revenue beating all estimates.")
print(f"label={label}, emb shape={emb.shape}, finite={np.all(np.isfinite(emb))}")

## Compare base vs fine-tuned on sample sentences

In [ ]:
base_enc = SentimentEncoder()  # default ProsusAI/finbert
ft_enc = SentimentEncoder(model_name_or_path=str(output_dir))

sentences = [
    "Strong earnings beat expectations across all segments.",
    "Company files for bankruptcy protection amid mounting losses.",
    "The board approved the quarterly dividend payment.",
    "Revenue fell short of analyst estimates for the third consecutive quarter.",
    "The acquisition is expected to close in Q3, subject to regulatory approval.",
]

print(f"{'base':>6}  {'ft':>4}  sentence")
print("-" * 70)
for s in sentences:
    b_label, _ = base_enc.encode(s)
    ft_label, _ = ft_enc.encode(s)
    marker = "" if b_label == ft_label else " <-- differs"
    print(f"{b_label:>6}  {ft_label:>4}  {s[:55]}{marker}")